# Evaluacion cuantitativa del RAG

Hasta este momento hemos realizado un agente que parece estar desenvolviendose correctamente. Sin embargo no podemos quedarnos con sensaciones, somos ingenieros y tenemos que medir todo, ya que lo que no se puede medir, no se puede gestionar, y lo que no se puede gestionar, no se puede mejorar

Es importante aclarar que nos vamos a remitir a evaluar unicamente el modulo de `Retrieval` del **RAG**, ya qu eintentar medir todo el comportamiento del agente, sumaria un ruido innecesario y nos desviaria del tema que elegimos de la consigna del proyecto para esta materia.

Lo que buscamos demostrar aquí es únicamente la capacidad matemática de nuestro índice vectorial **FAISS** para encontrar la información correcta: dada una alerta de Wazuh, ¿puede el sistema traer la técnica MITRE esperada entre sus primeros 5 resultados?

Para mantener el foco, el diseño de este experimento implica:

* `Aislar el Retriever:` Vamos a consultar directamente la base vectorial FAISS y la función de búsqueda por palabras clave, sin pasar por el Router ni ejecutar nodos del agente.

* `Ignorar la generación LLM:` No evaluaremos la redacción del reporte final ni las alucinaciones, solo validaremos los IDs de las técnicas (ej. T1059) recuperadas.

* `Métricas exactas:` Todo se reduce a tabular el **Hit@5** y el **MRR**, tal como solicita la corrección del profesor.

## Conceptos clave

Antes de pasar a medir, debemos tener bien claro algunos conceptos:

Para garantizar que nuestro agente brinde diagnósticos precisos, Capazeta utiliza una arquitectura **RAG** (Retrieval-Augmented Generation). Este enfoque permite al modelo de lenguaje fundamentar sus respuestas buscando contexto técnico directamente en el catálogo **MITRE ATT&CK**. El motor que hace posible esta recuperación de información es **FAISS**, una herramienta que indexa nuestros datos y realiza búsquedas vectoriales para encontrar las técnicas semánticamente más similares a la alerta que estamos analizando.

Sin embargo, para validar que esta arquitectura realmente funciona, necesitamos medir la precisión de nuestro módulo de recuperación. Para ello, evaluaremos matemáticamente los resultados de FAISS utilizando dos métricas estándar en la industria. Por un lado, utilizaremos **Hit@K** (específicamente Hit@5) para responder a una pregunta binaria: ¿logró el sistema traer la técnica MITRE correcta dentro de sus primeros 5 resultados? Por otro lado, calcularemos el **RR** (Mean Reciprocal Rank) para evaluar la calidad del ordenamiento, verificando qué tan cerca del primer lugar apareció esa técnica correcta.

Finalmente, para automatizar este proceso, registrar los experimentos y comparar el rendimiento de nuestro índice vectorial contra una simple búsqueda por palabras clave, orquestaremos toda la evaluación utilizando **LangSmith**.

## Setup

### Langmisth

#### ¿Por qué utilizamos LangSmith?
Para evaluar cuantitativamente nuestro sistema RAG, no basta con ejecutar consultas aisladas en la terminal. Necesitamos un entorno estructurado que nos permita medir el desempeño de forma repetible. Elegimos LangSmith para esta etapa por tres razones fundamentales:

* Gestión del Ground Truth: Nos permite cargar y versionar nuestro dataset de pruebas (las 20-30 alertas de Wazuh mapeadas manualmente a sus técnicas MITRE correctas).

* Automatización de las Métricas: Su motor de evaluación calcula nativamente el Hit@5 y el MRR ejecutando nuestras funciones de búsqueda iterativamente contra el dataset, evitando que codifiquemos bucles de cálculo manuales.

* Trazabilidad y Análisis de Errores: En los casos donde el sistema no logra un "Hit", la interfaz web de LangSmith captura la traza exacta de la ejecución, permitiéndonos inspeccionar qué documentos recuperó FAISS y entender por qué falló la similitud semántica.

A continuación, inicializamos el cliente y vinculamos nuestro notebook con la plataforma utilizando las variables de entorno de LangChain.

```bash
# LangSmith
LANGSMITH_TRACING=true
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
LANGSMITH_API_KEY= # api key generada en interfaz de langsmith
LANGSMITH_PROJECT=IA
```

Ahora simplemente con esto podemos ir a la interfaz de lang smith y ver que vemos nuestro proyecto correctamente
<div align="center">
  <img src="./img/02_evaluacion_rag/image.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Tambien en la seccion de *tracing* podemos ver diferentes aspectos, desde latencia hasta el tag que propusimos que nos dice que modelo hemos usado

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

## Carga del dataset (Ground Truth)

Lo que tenemos que hacer ahora es generar un dataset de prueba contra el cual vamos a contrastar los resultados del RAG y evaluarlo. Para hacer esto tendremos que hacer que se disparen una cierta cantidad de eventos que sean registrados en Wazuh y sean un mapeo claro a MITRE ATT&CK, por ejemplo realizaremos lo siguiente:

* `(T1110) Fuerza bruta SSH :` Intenta iniciar sesión por SSH varias veces con una contraseña incorrecta.

* `(T1046) Descubrimiento de red:` Ejecuta un escaneo con nmap localhost

* `(T1136) Creación de cuentas locales :` Añade un usuario de prueba ejecutando sudo useradd usuario_fantasma.

* `T1548 Abuso de privilegios :` Falla intencionalmente la autenticación de sudo o cambia permisos críticos de archivos en /etc (si tienes File Integrity Monitoring activado).

* `T1070 (Borrado de huellas):` Limpiar el historial de la terminal ejecutando history -c o intentando borrar un log con rm /var/log/auth.log.

* `T1053 (Tareas programadas):` Agregar un cron job persistente mediante crontab -e.

* `T1082 (Descubrimiento de sistema):` Ejecutar comandos de reconocimiento básicos como uname -a o cat /etc/os-release.

* `T1087 (Descubrimiento de cuentas):` Intentar leer el archivo de contraseñas protegidas con cat /etc/shadow.

* `T1049 (Descubrimiento de red):` Listar puertos y conexiones activas usando netstat -tunlp o ss -tuln.

* `T1222 (Modificación de permisos):` Cambiar los permisos de un archivo crítico a acceso público con chmod 777 /etc/passwd.

* `T1059 (Intérprete de comandos):` Ejecutar comandos ofuscados en base64 en la terminal.

### Generacion de las alertas

Primero nos aseguramos que estemos reportando correctamente al SOC

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 2.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Para realizar todo esto de una vez, se plantea ejecutar este script
```bash
#!/bin/bash
echo "Generando eventos de seguridad para Wazuh..."

# T1082: Descubrimiento de sistema
uname -a > /dev/null
cat /etc/os-release > /dev/null

# T1087: Descubrimiento de cuentas (requiere root)
cat /etc/shadow > /dev/null 2>&1

# T1049: Descubrimiento de red
ss -tuln > /dev/null

# T1222: Modificación de permisos
touch /tmp/wazuh_test_file.txt
chmod 777 /tmp/wazuh_test_file.txt

# T1136: Creación de cuentas locales temporales
useradd usuario_fantasma_wazuh 2>/dev/null
userdel usuario_fantasma_wazuh 2>/dev/null

# T1059: Ejecución en intérprete de comandos ofuscado (Base64)
echo "d2hvYW1p" | base64 -d | bash > /dev/null

# T1070: Borrado de huellas
history -c

# T1046: Escaneo rápido de red local
nc -vz localhost 1-1000 2>&1 > /dev/null

echo "Eventos generados. Revisa el dashboard de Wazuh."
```

El script se guardo dentro de `tests` y antes de ejecutarlo es necesario darle permisos de ejecucion: `chmod +x generador_alertas.sh` y ejecutarlo como `sudo`


<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 3.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 4.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>


### Export al csv

Vamos a la seccion de ``Threats Hunting > Events`` y vemos que efectivamente se han captado los logs que queriamos. Ahora tenemos que elegir los correctos y exportarlos en csv

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 6.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Antes de pasar a hacerlo, podemos ir a la vista de clasificacion de alertas en MITRE&ATACK del propio Wazuh para constatar cuantas de estas alertas pudo clasificar correctamente. Vemos que ha podido hacerlo unicamente con 4 alertas, por lo que justifica lo que estamos haciendo con el agente

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 5.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Podemos decir que el valor agregado de Capazeta frente a un SIEM convencional como Wazuh se resume en tres puntos principales:

* `Rigidez vs. Flexibilidad Semántica:` Wazuh clasifica eventos mediante reglas estáticas en XML. Si una regla detecta un evento (como un bloqueo de AppArmor o un cambio de puerto) pero el creador de la regla no le escribió explícitamente la etiqueta <mitre><id>TXXXX</id></mitre> en su configuración, el SIEM lo muestra como un evento "sin técnica MITRE". Capazeta resuelve esto analizando el significado en lenguaje natural del log mediante embeddings, sin depender de etiquetas preexistentes.

* `Cobertura de Alertas Genéricas:` En entornos de producción, la gran mayoría de los logs del sistema operativo (eventos de kernel, auditd, autenticaciones PAM) son genéricos. Wazuh los registra como nivel de severidad bajo o sin contexto de amenaza. Nuestro RAG es capaz de inferir qué técnica del catálogo MITRE ATT&CK está teniendo lugar aunque la alerta no haya sido diseñada específicamente para ese ataque.

* `Reducción del esfuerzo manual:` Evita que el equipo de SOC (Security Operations Center) tenga que mantener y actualizar manualmente miles de reglas XML de mapeo a medida que el catálogo de MITRE evoluciona.

Veamos el archivo exportado

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 7.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Como vemos existen eventos repetidos y la columa de mitre.id esta vacia en la mayoria.

Nuestro trabajo es elegir las 30 alertas relevantes y representativas, aasignandole una regla de mitre id a cada una

Al revisar el documento vemos que la mayoria de eventos se repiten y al final solo termianmos obteniendo 10 reglas para mapear. Esto es un problema ya que debemos darle una amplia variedad para testear nuestro RAG

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 8.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Lo que haremos ahora es buscar mas maneras de generar logs que mapeen a reglas MITRE con descripciones distintas.

Objetivos y Configuración (Setup)

    Texto: Breve introducción explicando que el objetivo es medir el desempeño del RAG de Capazeta (Hit@5 y MRR) comparado con una búsqueda clásica, cumpliendo con la auditoría del MVP.

    Código: Importación de librerías (pandas, langsmith, tu módulo rag, etc.) y carga de variables de entorno (.env), inicializando elo nmap -sS <tu_ip>. cliente de LangSmith.

2. Carga del Dataset (Ground Truth)

    Texto: Explicación de cómo se seleccionaron las 20-30 alertas representativas de Wazuh y cómo se etiquetaron manualmente con sus técnicas MITRE.

    Código: Carga del dataset (por ejemplo, leyendo un archivo alertas_wazuh_test.csv con pandas) y muestra del .head() para que el profesor vea la estructura (columna "alerta" y columna "tecnica_esperada").

3. Implementación del Baseline (Búsqueda por Palabras Clave)

    Texto: Justificación de la necesidad de un modelo de referencia (baseline) sin IA para medir la ganancia real del RAG.

    Código: Función simple que busque coincidencias de palabras clave de la alerta dentro del catálogo MITRE, y cálculo preliminar de cuántas veces le acierta en el top 5.

4. Evaluación del RAG de Capazeta (FAISS)

    Texto: Descripción del proceso de evaluación del sistema semántico actual.

    Código: Ejecución de tu cadena RAG sobre las alertas del dataset, registrando los resultados y trazabilidad en LangSmith.

5. Cálculo de Métricas y Comparación (Hit@5 y MRR)

    Texto: Definición técnica de qué son Hit@5 y MRR.

    Código: Tabla comparativa (DataFrame) o un gráfico de barras (usando matplotlib o seaborn) mostrando el rendimiento del Baseline vs. RAG.

6. Conclusiones y Análisis de Errores

    Texto: Párrafo final resumiendo qué método fue superior y analizando por qué el RAG falló en los casos donde no obtuvo un "Hit" (ej. alertas demasiado ambiguas).